# LCEL 인터페이스

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. `invoke`, `batch`, `stream` 동기 메서드의 차이를 설명하고 적절한 상황에 사용할 수 있어요
2. `ainvoke`, `abatch`, `astream` 비동기 메서드를 Jupyter 환경에서 실행할 수 있어요
3. `asyncio.gather`로 여러 LLM 호출을 병렬 실행해 성능을 높일 수 있어요

## 사전 지식

- `02_Chain.ipynb`의 파이프 연산자와 체인 구성 방법
- Python `async`/`await` 기초 (선택, 섹션 2에서 사용해요)

## 이전 노트북 연결

`02_Chain.ipynb`에서 체인을 구성하는 방법을 배웠어요. 이번에는 그 체인을 다양한 방식으로 실행하는 LCEL(LangChain Expression Language) 표준 인터페이스를 살펴볼게요.

아래 다이어그램은 LCEL 파이프라인에서 파이프 연산자가 데이터를 어떻게 전달하는지 보여줘요.

```mermaid
flowchart LR
    A["입력<br/>{question}"]:::input --> B["PromptTemplate<br/>|"]:::process
    B --> C["ChatOpenAI<br/>|"]:::process
    C --> D["StrOutputParser<br/>최종 출력"]:::output

    A2["invoke<br/>단일 실행"]:::process --> R1["결과 1개"]:::output
    A3["batch<br/>리스트 입력"]:::process --> R2["결과 리스트"]:::output
    A4["stream<br/>청크 스트리밍"]:::process --> R3["토큰 단위<br/>실시간 출력"]:::output
    A5["ainvoke / abatch<br/>/ astream"]:::process --> R4["비동기 결과"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef error fill:#f8d7da,stroke:#dc3545,color:#721c24
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
    classDef external fill:#d1ecf1,stroke:#17a2b8,color:#0c5460
```

`Runnable(러너블)` 프로토콜은 LangChain의 대부분 컴포넌트에 구현되어 있어요. 표준 인터페이스를 통해 사용자 정의 체인을 정의하고 일관된 방식으로 호출할 수 있어요.

**동기 메서드**:
- `invoke`: 단일 입력을 처리하고 결과를 반환해요
- `batch`: 여러 입력을 일괄 처리해요
- `stream`: 응답 청크를 실시간으로 스트리밍해요

**비동기 메서드**:
- `ainvoke`, `abatch`, `astream`: 위 메서드의 비동기 버전이에요
- `astream_log`: 최종 응답뿐만 아니라 중간 단계도 스트리밍해요

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import asyncio
import time

load_dotenv()

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

# 기본 체인 구성: 프롬프트 → LLM → 문자열 추출
# 아래 섹션에서 invoke, batch, stream 등 다양한 방식으로 이 체인을 호출해요
prompt_template = PromptTemplate.from_template("{question}")
chain = prompt_template | model | StrOutputParser()

## 1. 동기 메서드 (Synchronous Methods)

동기 메서드는 각 작업이 완료될 때까지 다음 작업을 기다려요. 코드 흐름이 단순하고 예측하기 쉬워요.

세 가지 동기 메서드를 순서대로 살펴볼게요: `invoke` → `batch` → `stream`

### 1.1 invoke - 단일 입력 처리

`invoke`는 가장 기본적인 메서드예요. 단일 입력에 대해 체인을 실행하고 결과를 반환해요. 응답이 완전히 완료된 뒤에야 반환돼요.

In [ ]:
# ---------------------------------------------------
# invoke(): 단일 입력 처리하고 완성된 응답 반환하기
# ---------------------------------------------------
# invoke()는 응답 전체가 완성될 때까지 블로킹 후 결과를 반환해요


### 1.2 batch - 여러 입력 일괄 처리

`batch`는 입력 리스트를 받아 내부적으로 병렬 처리해 결과 리스트를 반환해요. 개별 `invoke()`를 여러 번 호출하는 것보다 빠를 수 있어요.

In [ ]:
# ---------------------------------------------------
# batch(): 여러 입력을 리스트로 전달해 한 번에 처리하기
# ---------------------------------------------------
# ============================================================
# TODO: questions 리스트에 원하는 질문을 추가하고 batch()로 실행해봐요
# 힌트: chain.batch([{"question": "..."}, ...]) 형태로 호출해요
# 예상 결과: 각 질문에 대한 응답이 리스트로 반환돼요
# ============================================================


### 1.3 stream - 스트리밍 응답

`stream`은 LLM이 토큰을 생성할 때마다 즉시 청크(chunk)를 전달해요. 긴 응답을 처리할 때 사용자가 첫 결과를 더 빨리 볼 수 있어요.

> **실무 팁**: 챗봇 UI처럼 응답을 실시간으로 표시해야 하는 경우 `stream`을 사용해요. 배치 처리나 응답 전체를 저장해야 할 때는 `invoke`가 더 적합해요.

In [ ]:
# ---------------------------------------------------
# stream(): 토큰이 생성되는 즉시 화면에 출력하기
# ---------------------------------------------------
# ChatGPT 웹 화면에서 글자가 한 글자씩 나타나는 것과 같은 원리예요
# 응답 완성을 기다리지 않으므로 사용자 체감 속도가 향상돼요


### 1.4 stream - 청크 수집해 전체 메시지 구성

스트리밍 청크를 변수에 누적하면 전체 응답도 함께 얻을 수 있어요. 실시간 출력과 전체 저장을 동시에 처리하는 패턴이에요.

In [ ]:
# ---------------------------------------------------
# 스트리밍 청크를 누적해 전체 응답 함께 구성하기
# ---------------------------------------------------
# 실시간 출력과 전체 텍스트 저장을 동시에 처리하는 패턴이에요


## 2. 비동기 메서드 (Asynchronous Methods)

앞서 동기 메서드를 살펴봤어요. 이제 비동기(async) 메서드를 알아볼게요.

LLM API 호출 시간의 대부분은 **네트워크 응답 대기**예요. 동기 방식에서는 대기하는 동안 아무것도 못 하지만, 비동기 방식에서는 그 시간에 다른 API 호출을 시작할 수 있어요.

| 방식 | 동작 |
|------|------|
| 동기 `invoke` 3회 | A 완료 대기 → B 완료 대기 → C 완료 대기 (총 시간 = A + B + C) |
| 비동기 `ainvoke` 3회 (`asyncio.gather`) | A, B, C 동시 시작 → 가장 느린 것만큼 대기 (총 시간 ≈ max(A, B, C)) |

> **실무 팁**: FastAPI 같은 웹 서버 환경에서는 비동기가 필수예요. 한 사용자의 LLM 응답을 기다리는 동안 다른 사용자 요청도 처리할 수 있기 때문이에요.

### 2.1 ainvoke - 비동기 단일 입력 처리

`ainvoke`는 `invoke`의 비동기 버전이에요. Jupyter 노트북은 이미 실행 중인 이벤트 루프가 있어서 `asyncio.run()` 대신 `await`를 셀에서 직접 사용할 수 있어요.

> **주의**: 일반 Python 스크립트에서는 `asyncio.run(chain.ainvoke(...))` 형태로 실행해야 해요. Jupyter에서만 `await`를 최상위 레벨에서 사용할 수 있어요.

In [ ]:
# ---------------------------------------------------
# ainvoke(): 비동기 단일 호출 (Jupyter에서 await 직접 사용)
# ---------------------------------------------------
# Jupyter 노트북은 이미 이벤트 루프가 실행 중이므로 await를 셀 최상위에서 사용 가능
# 일반 Python 스크립트에서는 asyncio.run(chain.ainvoke(...)) 형태로 실행해야 해요


### 2.2 abatch - 비동기 일괄 처리

`abatch`는 `batch`의 비동기 버전이에요. 내부적으로 `asyncio.gather`를 사용해 여러 입력을 동시에 처리해요.

In [ ]:
# ---------------------------------------------------
# abatch(): 비동기 일괄 처리 (내부적으로 asyncio.gather 사용)
# ---------------------------------------------------


### 2.3 astream - 비동기 스트리밍

`astream`은 `stream`의 비동기 버전이에요. `async for`로 청크를 하나씩 받아요. 이벤트 루프가 블로킹되지 않아 웹소켓으로 실시간 전송하면서 다른 작업을 동시에 수행할 수 있어요.

In [ ]:
# ---------------------------------------------------
# astream(): 비동기 스트리밍 (async for로 청크 수신)
# ---------------------------------------------------
# astream()은 이벤트 루프를 블로킹하지 않아요
# 웹소켓으로 사용자에게 실시간 전송하면서 다른 비동기 작업을 동시에 수행 가능해요


## 3. 동기 vs 비동기 성능 비교

비동기 메서드를 살펴봤어요. 이제 실제로 시간을 측정해 동기와 비동기 방식의 차이를 직접 확인해볼게요.

> **참고**: `batch()`와 `abatch()`는 모두 내부적으로 최적화가 되어 있어 성능 차이가 크지 않을 수 있어요. 병렬 처리의 이점을 명확히 보려면 개별 `invoke` 순차 호출과 개별 `ainvoke` 병렬 호출을 비교해야 해요.

---

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

| 메서드 | 설명 | 사용 상황 |
|--------|------|-----------|
| `invoke` | 단일 입력, 응답 완료 후 반환 | 단순 단일 호출 |
| `batch` | 여러 입력 일괄 처리 | 다수 입력을 동시에 처리할 때 |
| `stream` | 토큰 단위 실시간 출력 | 챗봇 UI, 긴 응답 실시간 표시 |
| `ainvoke` | 비동기 단일 호출 | 웹 서버, 비동기 환경 |
| `abatch` | 비동기 일괄 처리 | 비동기 환경에서 다수 입력 |
| `astream` | 비동기 스트리밍 | 웹소켓 실시간 전송 |

## 다음 노트북 예고

다음 `04_Runnable_Basic.ipynb`에서는 `RunnablePassthrough`, `RunnableParallel`, `RunnableLambda` 세 가지 핵심 Runnable을 조합해 복잡한 파이프라인을 구성하는 방법을 배울 거예요.

In [ ]:
# ---------------------------------------------------
# 동기 vs 비동기 실행 시간 비교하기
# ---------------------------------------------------
# ============================================================
# TODO: 아래 코드를 실행하고 순차 invoke vs 병렬 ainvoke 시간 차이를 확인해요
# 힌트: asyncio.gather()로 여러 코루틴을 동시에 실행해요
# 예상 결과: 비동기 병렬 방식이 순차 방식보다 약 2~3배 빠를 거예요
# ============================================================
